In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import pickle

/Users/dylan/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/dylan/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
fbref_url = "https://fbref.com{}"
premier_league = fbref_url.format("/en/comps/9/Premier-League-Stats")
data = requests.get(premier_league)

## Get All Matches

In [3]:
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
import logging
import pandas as pd

NEO4J_URI='neo4j+s://2b521897.databases.neo4j.io'
NEO4J_USERNAME='neo4j'
NEO4J_PASSWORD='nCUihZktDGHjJUpQtKUal0IJhrk0eSuMdSFUm99Th3Q'
AURA_INSTANCEID='2b521897'
AURA_INSTANCENAME='Instance01'

In [4]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [5]:
with driver.session(database="neo4j") as session:
    query = "CREATE (l:League {name: $name})"
    session.run(query, {"name":"PremierLegue24-25"})

In [6]:
with open(f'team_dict_24-25.pkl', 'rb') as f:
        team_dict = pickle.load(f)

In [7]:
team_players = dict()
player_details = list()
for team, players in team_dict.items():
    team_players[team] = list()
    for player in players.keys():
        team_players[team].append(player)
        players[player]['Name'] = player
        player_details.append(players[player])

In [8]:
with driver.session(database="neo4j") as session:
    query = """
            UNWIND $players AS player
            MERGE (p:Player {name: player.Name})
            SET p.position = player.position,
                p.footed = player.footed,
                p.height = player.height,
                p.weight = player.weight,
                p.dob = player.dob,
                p.birth_place = player.birth_place,
                p.national_team = player.national_team,
                p.club = player.club,
                p.wages = player.wages,
                p.contract_expiry = player.contract_expiry,
                p.img = player.img
    """
    
    session.run(query, players=player_details)

In [9]:
# clean up before sending
with open('match_reports.pkl','rb') as f:
    match_reports = pickle.load(f)

In [22]:
fixtures_df = pd.read_csv('fixtures.csv')
fixtures_df[['Home_Score', 'Away_Score']] = fixtures_df['Score'].str.split('–', expand=True)

In [11]:
all_players_dict = pd.read_csv('all_players.csv', index_col=0).to_dict(orient='index')

In [12]:
# this is very important please do not delete
mappings = {"Brighton":'Brighton & Hove Albion',
            'Manchester Utd':'Manchester United',
            'Newcastle Utd':'Newcastle United',
            "Nott'ham Forest":'Nottingham Forest',
            'Tottenham':'Tottenham Hotspur',
            'West Ham':'West Ham United',
            'Wolves':'Wolverhampton Wanderers'}

In [23]:
fixtures_df['Home'] = fixtures_df['Home'].apply(lambda x: mappings[x] if x in mappings else x)
fixtures_df['Away'] = fixtures_df['Away'].apply(lambda x: mappings[x] if x in mappings else x)

In [14]:
with driver.session(database="neo4j") as session:
    query = "match (n) detach delete n"
    session.run(query)

In [15]:
with driver.session(database="neo4j") as session:
    query = "CREATE (l:League {name: $name})"
    session.run(query, {"name":"PremierLeague"})

    query = "CREATE (s:Season {name: $name})"
    session.run(query, {"name":"24-25"})

    query = """
        MATCH (s:Season{name:$season_name}), (l:League{name: $league_name}) 
        CREATE (s)-[:SEASON_OF_LEAGUE]->(l)
        """
    print(session.run(query, {"league_name": "PremierLeague", "season_name": "24-25"}))

In [16]:
team_players = dict()
player_details = list()

for team, players in team_dict.items():
    team_players[team] = list()
    for player in players.keys():
        team_players[team].append(player)
        players[player]['Name'] = player
        players[player]['Team'] = team  # Fix the key access from player to players
        player_details.append(players[player])
    team_players[team] = team_players[team]

with driver.session(database="neo4j") as session:
    query = """
            UNWIND $players AS player
            MERGE (p:Player {name: player.Name})
            SET p.position = player.position,
                p.footed = player.footed,
                p.height = player.height,
                p.weight = player.weight,
                p.dob = player.dob,
                p.birth_place = player.birth_place,
                p.national_team = player.national_team,
                p.club = player.club,
                p.wages = player.wages,
                p.contract_expiry = player.contract_expiry,
                p.img = player.img

            MERGE (team:Team {name: player.Team})
            MERGE (p)-[:PLAYS_FOR]->(team)
    """
    
    session.run(query, players=player_details)


In [17]:
with driver.session(database="neo4j") as session:
    # Create GameWeek nodes and relationships
    for gw in range(1, 39):  # Loop from 1 to 38
        query = """
        MATCH (s:Season {name: $season_name})
        CREATE (gw:GameWeek {number: $number})-[:IS_A_GW]->(s)
        """
        session.run(query, {"season_name": "24-25", "number": gw})

In [24]:
with driver.session(database="neo4j") as session:
    for _, row in fixtures_df.iterrows():
        query = """
        MERGE (match:Match {id: $match_id})
        SET match.date = $date,
            match.time = $time,
            match.home_team = $home_team,
            match.away_team = $away_team,
            match.home_expected = $home_expected,
            match.away_expected = $away_expected,
            match.home_score = $home_score,
            match.away_score = $away_score

        MERGE (ref:Referee {name: $referee})
        MERGE (venue:Venue {name: $venue})
        
        MERGE (home_team:Team {name: $home_team})
        MERGE (away_team:Team {name: $away_team})
        
        MERGE (match)-[:REFERRED_BY]->(ref)
        MERGE (match)-[:PLAYED_AT]->(venue)
        MERGE (match)-[:HOME_TEAM]->(home_team)
        MERGE (match)-[:AWAY_TEAM]->(away_team)
        """
        
        # Prepare the data dictionary
        data ={
            "match_id": f"GW{row['GW']}_{row['Home']}_vs_{row['Away']}",
            "date": row["Date"],
            "time": row["Time"],
            "home_team": row["Home"],
            "away_team": row["Away"],
        }
        if row["Match Report"]:
            data["home_expected"]= row["xG"],
            data["away_expected"]= row["xG.1"],
            data["home_score"]= row["Home_Score"],
            data["away_score"]= row["Away_Score"],
            data["referee"]= row["Referee"],
            data["venue"] =  row["Venue"]
        
        # Execute the query
        session.run(query, data)


In [19]:
fixtures_df

,Unnamed: 0,GW,Day,Date,Time,Home,xG,Score,xG.1,Away,Attendance,Venue,Referee,Match Report,Notes,Home_Score,Away_Score
0,0,GW1,Fri,2024-08-16,20:00,Manchester United,2.4,1–0,0.4,NaN,73297.0,Old Trafford,Robert Jones,Match Report,NaN,1,0
1,1,GW1,Sat,2024-08-17,12:30,NaN,0.5,0–2,2.6,NaN,30014.0,Portman Road Stadium,Tim Robinson,Match Report,NaN,0,2
2,2,GW1,Sat,2024-08-17,15:00,Newcastle United,0.3,1–0,1.8,NaN,52196.0,St James' Park,Craig Pawson,Match Report,NaN,1,0
3,3,GW1,Sat,2024-08-17,15:00,Nottingham Forest,1.3,1–1,1.2,NaN,29763.0,The City Ground,Michael Oliver,Match Report,NaN,1,1
4,4,GW1,Sat,2024-08-17,15:00,NaN,0.5,0–3,1.4,Brighton & Hove Albion,39217.0,Goodison Park,Simon Hooper,Match Report,NaN,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,413,GW38,Sun,2025-05-25,16:00,NaN,NaN,NaN,NaN,NaN,NaN,Craven Cottage,NaN,Head-to-Head,NaN,NaN,NaN
376,414,GW38,Sun,2025-05-25,16:00,Nottingham Forest,NaN,NaN,NaN,NaN,NaN,The City Ground,NaN,Head-to-Head,NaN,NaN,NaN
377,415,GW38,Sun,2025-05-25,16:00,Manchester United,NaN,NaN,NaN,NaN,NaN,Old Trafford,NaN,Head-to-Head,NaN,NaN,NaN
378,416,GW38,Sun,2025-05-25,16:00,Wolverhampton Wanderers,NaN,NaN,NaN,NaN,NaN,Molineux Stadium,NaN,Head-to-Head,NaN,NaN,NaN


In [35]:
match_reports[0][0][1]

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Unnamed: 3_level_0  \
              Player             Nation                Age                Min   
0        André Onana             cm CMR             28-136                 90   

  Shot Stopping                      Launched  ...  Passes        Goal Kicks  \
           SoTA GA Saves  Save% PSxG      Cmp  ... Launch% AvgLen        Att   
0             2  0     2  100.0  0.9        3  ...    34.5   29.6          0   

                 Crosses           Sweeper          
  Launch% AvgLen     Opp Stp  Stp%    #OPA AvgDist  
0     NaN    NaN      16   2  12.5       4    16.3  

[1 rows x 24 columns]

In [39]:
with driver.session(database="neo4j") as session:
    for i, row in fixtures_df[:len(match_reports)].iterrows():
        match_id = f"GW{row['GW']}_{row['Home']}_vs_{row['Away']}"

        home_team_stats = match_reports[i][0][0][:-1]
        away_team_stats = match_reports[i][1][0][:-1]

        home_team_goaly_stats = match_reports[i][0][1]
        away_team_goaly_stats = match_reports[i][1][1]
        
        home_team_name = row['Home']
        away_team_name = row['Away']

        # Add Home Team Player Stats
        for _, player_stats in home_team_stats.iterrows():
            player_query = """
            MERGE (player:Player {name: $player_name, position: $position, age: $age})
            MERGE (team:Team {name: $team_name})
            MERGE (player)-[:PLAYS_FOR]->(team)
            MERGE (match:Match {id: $match_id})
            MERGE (match)-[h:HOME_TEAM_PLAYER_STATS]->(player)
            SET h.goals = $goals,
                h.assists = $assists,
                h.penalties = $penalties,
                h.shots = $shots,
                h.shots_on_target = $shots_on_target,
                h.touches = $touches,
                h.tackles = $tackles,
                h.interceptions = $interceptions,
                h.blocks = $blocks,
                h.xg = $xg,
                h.npxg = $npxg,
                h.xag = $xag,
                h.sca = $sca,
                h.gca = $gca,
                h.passes_completed = $passes_completed,
                h.passes_attempted = $passes_attempted,
                h.pass_accuracy = $pass_accuracy,
                h.progressive_passes = $progressive_passes,
                h.carries = $carries,
                h.progressive_carries = $progressive_carries
            """
            session.run(player_query, {
                "player_name": player_stats[('Unnamed: 0_level_0', 'Player')],
                "position": player_stats[('Unnamed: 3_level_0', 'Pos')],
                "age": player_stats[('Unnamed: 4_level_0', 'Age')],
                "match_id": match_id,
                "team_name": home_team_name,
                "goals": player_stats[('Performance', 'Gls')],
                "assists": player_stats[('Performance', 'Ast')],
                "penalties": player_stats[('Performance', 'PK')],
                "shots": player_stats[('Performance', 'Sh')],
                "shots_on_target": player_stats[('Performance', 'SoT')],
                "touches": player_stats[('Performance', 'Touches')],
                "tackles": player_stats[('Performance', 'Tkl')],
                "interceptions": player_stats[('Performance', 'Int')],
                "blocks": player_stats[('Performance', 'Blocks')],
                "xg": player_stats[('Expected', 'xG')],
                "npxg": player_stats[('Expected', 'npxG')],
                "xag": player_stats[('Expected', 'xAG')],
                "sca": player_stats[('SCA', 'SCA')],
                "gca": player_stats[('SCA', 'GCA')],
                "passes_completed": player_stats[('Passes', 'Cmp')],
                "passes_attempted": player_stats[('Passes', 'Att')],
                "pass_accuracy": player_stats[('Passes', 'Cmp%')],
                "progressive_passes": player_stats[('Passes', 'PrgP')],
                "carries": player_stats[('Carries', 'Carries')],
                "progressive_carries": player_stats[('Carries', 'PrgC')],
            })

        # Add Away Team Player Stats
        for _, player_stats in away_team_stats.iterrows():
            player_query = """
            MERGE (player:Player {name: $player_name, position: $position, age: $age})
            MERGE (team:Team {name: $team_name})
            MERGE (player)-[:PLAYS_FOR]->(team)
            MERGE (match:Match {id: $match_id})
            MERGE (match)-[a:AWAY_TEAM_PLAYER_STATS]->(player)
            SET a.goals = $goals,
                a.assists = $assists,
                a.penalties = $penalties,
                a.shots = $shots,
                a.shots_on_target = $shots_on_target,
                a.touches = $touches,
                a.tackles = $tackles,
                a.interceptions = $interceptions,
                a.blocks = $blocks,
                a.xg = $xg,
                a.npxg = $npxg,
                a.xag = $xag,
                a.sca = $sca,
                a.gca = $gca,
                a.passes_completed = $passes_completed,
                a.passes_attempted = $passes_attempted,
                a.pass_accuracy = $pass_accuracy,
                a.progressive_passes = $progressive_passes,
                a.carries = $carries,
                a.progressive_carries = $progressive_carries
            """
            session.run(player_query, {
                "player_name": player_stats[('Unnamed: 0_level_0', 'Player')],
                "position": player_stats[('Unnamed: 3_level_0', 'Pos')],
                "age": player_stats[('Unnamed: 4_level_0', 'Age')],
                "match_id": match_id,
                "team_name": away_team_name,
                "goals": player_stats[('Performance', 'Gls')],
                "assists": player_stats[('Performance', 'Ast')],
                "penalties": player_stats[('Performance', 'PK')],
                "shots": player_stats[('Performance', 'Sh')],
                "shots_on_target": player_stats[('Performance', 'SoT')],
                "touches": player_stats[('Performance', 'Touches')],
                "tackles": player_stats[('Performance', 'Tkl')],
                "interceptions": player_stats[('Performance', 'Int')],
                "blocks": player_stats[('Performance', 'Blocks')],
                "xg": player_stats[('Expected', 'xG')],
                "npxg": player_stats[('Expected', 'npxG')],
                "xag": player_stats[('Expected', 'xAG')],
                "sca": player_stats[('SCA', 'SCA')],
                "gca": player_stats[('SCA', 'GCA')],
                "passes_completed": player_stats[('Passes', 'Cmp')],
                "passes_attempted": player_stats[('Passes', 'Att')],
                "pass_accuracy": player_stats[('Passes', 'Cmp%')],
                "progressive_passes": player_stats[('Passes', 'PrgP')],
                "carries": player_stats[('Carries', 'Carries')],
                "progressive_carries": player_stats[('Carries', 'PrgC')],
            })

         # Add Home Team Goalie Stats
        for _, goalie_stats in home_team_goaly_stats.iterrows():
            goalie_query = """
            MERGE (goalie:Player {name: $goalie_name, position: "GK", age: $age})
            MERGE (team:Team {name: $team_name})
            MERGE (goalie)-[:PLAYS_FOR]->(team)
            MERGE (match:Match {id: $match_id})
            MERGE (match)-[:HOME_TEAM_GOALIE_STATS]->(goalie)
            SET goalie.minutes_played = $minutes_played,
                goalie.sota = $sota,
                goalie.goals_allowed = $goals_allowed,
                goalie.saves = $saves,
                goalie.save_percentage = $save_percentage,
                goalie.psxg = $psxg,
                goalie.passes_completed = $passes_completed,
                goalie.passes_attempted = $passes_attempted,
                goalie.pass_accuracy = $pass_accuracy,
                goalie.gk_passes_attempted = $gk_passes_attempted,
                goalie.throws = $throws,
                goalie.launch_percentage = $launch_percentage,
                goalie.launch_average_length = $launch_average_length,
                goalie.goal_kicks_attempted = $goal_kicks_attempted,
                goalie.goal_kick_launch_percentage = $goal_kick_launch_percentage,
                goalie.goal_kick_average_length = $goal_kick_average_length,
                goalie.crosses_opportunities = $crosses_opportunities,
                goalie.crosses_stops = $crosses_stops,
                goalie.crosses_stop_percentage = $crosses_stop_percentage,
                goalie.opa = $opa,
                goalie.opa_average_distance = $opa_average_distance
            """
            session.run(goalie_query, {
                "goalie_name": goalie_stats[('Unnamed: 0_level_0', 'Player')],
                "age": goalie_stats[('Unnamed: 2_level_0', 'Age')],
                "match_id": match_id,
                "team_name": home_team_name,
                "minutes_played": goalie_stats[('Unnamed: 3_level_0', 'Min')],
                "sota": goalie_stats[('Shot Stopping', 'SoTA')],
                "goals_allowed": goalie_stats[('Shot Stopping', 'GA')],
                "saves": goalie_stats[('Shot Stopping', 'Saves')],
                "save_percentage": goalie_stats[('Shot Stopping', 'Save%')],
                "psxg": goalie_stats[('Shot Stopping', 'PSxG')],
                "passes_completed": goalie_stats[('Launched', 'Cmp')],
                "passes_attempted": goalie_stats[('Launched', 'Att')],
                "pass_accuracy": goalie_stats[('Launched', 'Cmp%')],
                "gk_passes_attempted": goalie_stats[('Passes', 'Att (GK)')],
                "throws": goalie_stats[('Passes', 'Thr')],
                "launch_percentage": goalie_stats[('Passes', 'Launch%')],
                "launch_average_length": goalie_stats[('Passes', 'AvgLen')],
                "goal_kicks_attempted": goalie_stats[('Goal Kicks', 'Att')],
                "goal_kick_launch_percentage": goalie_stats[('Goal Kicks', 'Launch%')],
                "goal_kick_average_length": goalie_stats[('Goal Kicks', 'AvgLen')],
                "crosses_opportunities": goalie_stats[('Crosses', 'Opp')],
                "crosses_stops": goalie_stats[('Crosses', 'Stp')],
                "crosses_stop_percentage": goalie_stats[('Crosses', 'Stp%')],
                "opa": goalie_stats[('Sweeper', '#OPA')],
                "opa_average_distance": goalie_stats[('Sweeper', 'AvgDist')],
            })

        # Add Away Team Goalie Stats
        for _, goalie_stats in away_team_goaly_stats.iterrows():
            session.run(goalie_query, {
                "goalie_name": goalie_stats[('Unnamed: 0_level_0', 'Player')],
                "age": goalie_stats[('Unnamed: 2_level_0', 'Age')],
                "match_id": match_id,
                "team_name": away_team_name,
                "minutes_played": goalie_stats[('Unnamed: 3_level_0', 'Min')],
                "sota": goalie_stats[('Shot Stopping', 'SoTA')],
                "goals_allowed": goalie_stats[('Shot Stopping', 'GA')],
                "saves": goalie_stats[('Shot Stopping', 'Saves')],
                "save_percentage": goalie_stats[('Shot Stopping', 'Save%')],
                "psxg": goalie_stats[('Shot Stopping', 'PSxG')],
                "passes_completed": goalie_stats[('Launched', 'Cmp')],
                "passes_attempted": goalie_stats[('Launched', 'Att')],
                "pass_accuracy": goalie_stats[('Launched', 'Cmp%')],
                "gk_passes_attempted": goalie_stats[('Passes', 'Att (GK)')],
                "throws": goalie_stats[('Passes', 'Thr')],
                "launch_percentage": goalie_stats[('Passes', 'Launch%')],
                "launch_average_length": goalie_stats[('Passes', 'AvgLen')],
                "goal_kicks_attempted": goalie_stats[('Goal Kicks', 'Att')],
                "goal_kick_launch_percentage": goalie_stats[('Goal Kicks', 'Launch%')],
                "goal_kick_average_length": goalie_stats[('Goal Kicks', 'AvgLen')],
                "crosses_opportunities": goalie_stats[('Crosses', 'Opp')],
                "crosses_stops": goalie_stats[('Crosses', 'Stp')],
                "crosses_stop_percentage": goalie_stats[('Crosses', 'Stp%')],
                "opa": goalie_stats[('Sweeper', '#OPA')],
                "opa_average_distance": goalie_stats[('Sweeper', 'AvgDist')],
            })


    Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
                Player                  #             Nation   
0      Bruno Fernandes                8.0             pt POR   
1      Marcus Rashford               10.0            eng ENG   
2          Amad Diallo               16.0             ci CIV   
3   Alejandro Garnacho               17.0             ar ARG   
4          Mason Mount                7.0            eng ENG   
5       Joshua Zirkzee               11.0             nl NED   
6        Kobbie Mainoo               37.0            eng ENG   
7      Scott McTominay               39.0            sct SCO   
8             Casemiro               18.0             br BRA   
9          Diogo Dalot               20.0             pt POR   
10   Lisandro Martínez                6.0             ar ARG   
11       Harry Maguire                5.0            eng ENG   
12         Jonny Evans               35.0            nir NIR   
13   Noussair Mazraoui                3.

In [ ]:
query = """
MATCH (player:Player)-[:PLAYS_FOR]->(team:Team)
WITH player, COUNT(team) AS teamCount
WHERE teamCount > 3
RETURN player, teamCount
"""
players_errors = list()
with driver.session(database="neo4j") as session:
    r = session.run(query)
    for record in r:
        players_errors.append(record["player"]["name"])